In [1]:
#0. 最初要改变的变量
Topology_Version = 'gridx'
P=18
N=36

In [2]:
# ====================== 环境准备 ======================
# 1) 设定环境变量（必须在导入 matplotlib 之前执行）
import os




os.environ['QT_API'] = 'pyqt5'        # 指定使用 PyQt5 作为 Qt 绑定
os.environ['MPLBACKEND'] = 'QtAgg'    # 指定 Matplotlib 后端为 QtAgg（更推荐，替代 TkAgg）

# 2) 启动 Qt 事件循环（控制台模式下，让 Qt 窗口能实时响应）
%gui qt5

# 3) 检查 matplotlib backend
import matplotlib as mpl
mpl.rcParams.update({
    # 这里按系统常见字体给一串候选，存在则自动生效
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "SimSun",
                        "Noto Sans CJK SC", "Source Han Sans SC",
                        "Arial Unicode MS", "DejaVu Sans"],
    "font.family": "sans-serif",
    "axes.unicode_minus": False,   # 负号用正常字符，避免被当作缺字形
})

print("backend (before pyplot):", mpl.get_backend())
# 如果不是 QtAgg，强制改为 QtAgg（注意：必须在导入 pyplot 前设置）
mpl.rcParams['backend'] = 'QtAgg'

# 4) 现在再导入 pyplot
import matplotlib.pyplot as plt
print("backend (after pyplot):", mpl.get_backend())





backend (before pyplot): QtAgg
backend (after pyplot): qtagg


In [5]:
from pathlib import Path
from src.viz.pyqt_main2 import SatelliteViewer
DATA_DIR = Path(r"D:\paper3")
BASEDIR =  DATA_DIR / "data"


Topology_DIR = 'topology_design'
Topology_Version = 'gridx'
TOPO_CFG_PATH=BASEDIR / Topology_DIR / Topology_Version/"config" / "motif.json"

xml_file = BASEDIR /'basic_file'/ 'satellitesposition' / "station_visible_satellites_20250106.xml"

In [6]:
import draw.read_snap_xml  as read_snap_xml
from src.config.viewer_config import ViewerConfig, G60_CONFIG

# 当我们修改df的时候，实际上，下面的是无需去修改的
# df 就是你前面已经读好的 DataFrame



# 2) 时间窗直接用 df 的范围，不要整天都画
WIN_START = 0
WIN_END = 86164

# 区域定义（从 G60_CONFIG.station_groups 读取）
all_regions = {}
for gid, info in G60_CONFIG.station_groups.items():
    all_regions[gid] = info["stations"]

# 收集所有 station id
all_station_ids = sorted(set(
    sid for stations in all_regions.values() for sid in stations
))

# 如果你要严格用 1..11 和 12..21，请改成：
# region_a_stations = list(range(1, 12))
# region_b_stations = list(range(12, 22))



series_list = read_snap_xml.parse_station_timeseries(
    xml_file, all_station_ids, WIN_START, WIN_END - 1
)
series_by_station = {sid: ts for sid, ts in zip(all_station_ids, series_list)}


# 数据处理
这一章节，我们要开始数据后处理，就是上面数据导出后，我们需要进行同轨异轨的一个分析验证


In [7]:

# 2) 确保 viewer 有全局引用，避免 GC 回收导致崩溃
if not hasattr(sys.modules[__name__], "_viewer_list"):
    _viewer_list = []
DATA_DIR = Path(r"D:\paper3")
BASEDIR =  DATA_DIR / "data"

Topology_DIR = 'topology_design'

In [41]:
Topology_Version = 'grid_x_sparse'

In [42]:
import  src.paper3_postprocess.read_path_csv as read_path_csv
# data_DIR = Path(FIGURE_DIR) / "region1_to_region2_0_100"   # 你导出的原始 csv 目录
FIGURE_DIR = BASEDIR / Topology_DIR / Topology_Version/"path"

data_DIR = Path(FIGURE_DIR) / "region_pairs_0_86164_route2"   # 你导出的原始 csv 目录


df = read_path_csv.read_pair_csv(data_DIR / "region1--station5-region2--station19.csv")


In [182]:
import  src.paper3_postprocess.read_path_csv as read_path_csv
# data_DIR = Path(FIGURE_DIR) / "region1_to_region2_0_100"   # 你导出的原始 csv 目录
FIGURE_DIR = BASEDIR / Topology_DIR / Topology_Version/"path"

data_DIR = Path(FIGURE_DIR) / "region_pairs_0_86164_minhop_v1"   # 你导出的原始 csv 目录


df = read_path_csv.read_pair_csv(data_DIR / "region1--station15-region2--station29.csv")


In [43]:
import src.model.basiclink as basiclink
from draw.basic_functio.topology_config import TopologyRecorder, load_config

cfg = load_config(BASEDIR / Topology_DIR / Topology_Version / "config" / "motif.json")
N = cfg.N
P = cfg.P

rec = TopologyRecorder(cfg.P, cfg.N)
rec._motifs = cfg.motifs

# 静态拓扑，只 render 一次
inter_adj = rec.render_adj_at(
    t=WIN_START,
    eval_env={"start_ts": WIN_START, "end_ts": WIN_END + 1}
)

raw_inter_once = inter_adj
raw_inter_once = basiclink.make_edges_bidirectional(raw_inter_once)

base_neighbors = {
    i * N + j: (i * N + ((j + 1) % N), i * N + ((j - 1) % N))
    for i in range(P) for j in range(N)
}
STATIC_EDGES = {node: {r, l} for node, (r, l) in base_neighbors.items()}
for src, dsts in raw_inter_once.items():
    STATIC_EDGES.setdefault(src, set()).update(dsts)


In [39]:
# probability 2d 原始图
# ========== 路径 → intra/inter 链路分类 ==========
# 星座参数
# P = 18
# N = 36
# 1) 从 df 里拿当前 station pair


df_s6_s8 = df
station_a = int(df_s6_s8["station_a"].iloc[0])
station_b = int(df_s6_s8["station_b"].iloc[0])
dfpath = df["path"]


import src.model.get_intra_inter_link as get_intra_inter_link


all_intra = []
all_inter = []

for idx, path_str in enumerate(df["path"]):
    intra, inter = get_intra_inter_link.parse_path_links(path_str, N=N)
    all_intra.append(intra)
    all_inter.append(inter)

# 写回 DataFrame
df["intra_links"] = all_intra    # 每行是 [(src,dst), ...] 的 list
df["inter_links"] = all_inter

# 同时统计跳数
df["intra_hops"] = df["intra_links"].apply(len)
df["inter_hops"] = df["inter_links"].apply(len)
df["total_hops"] = df["intra_hops"] + df["inter_hops"]
def plot_intra_inter_hops_over_time(
    df,
    *,
    time_col="time",
    intra_col="intra_hops",
    inter_col="inter_hops",
    figsize=(10, 4),
    title="Intra-orbit vs Inter-orbit hops over time",
    show=True,
    save=None,
    save_dir="figs",
    basename="intra_inter_hops",
    formats=("png", "pdf"),
    dpi=300,
    return_handles=True,
):
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    base_rc = {
        "font.family": "Times New Roman",
        "font.size": 14,
        "axes.labelsize": 18,
        "axes.titlesize": 18,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "axes.linewidth": 1.2,
    }

    plt.ion()
    with mpl.rc_context(base_rc):
        fig, ax = plt.subplots(figsize=figsize)

        t = df[time_col]

        ax.plot(t, df[intra_col], marker='o', markersize=3,
                linewidth=1.5, color='#2196F3', label='Intra-orbit hops')
        ax.plot(t, df[inter_col], marker='s', markersize=3,
                linewidth=1.5, color='#F44336', label='Inter-orbit hops')
        ax.plot(t, df[intra_col] + df[inter_col], marker='^', markersize=3,
                linewidth=1.2, color='#888888', linestyle='--', label='Total hops')

        ax.set_xlabel("Time Step")
        ax.set_ylabel("Hops")
        ax.set_title(title)
        ax.legend(framealpha=0.9)
        ax.grid(alpha=0.3, linestyle="--")
        plt.tight_layout()

        if show:
            plt.show(block=False)
            try:
                plt.pause(0.01)
            except Exception:
                pass

    if save:
        from pathlib import Path
        targets = []
        if save is True:
            outdir = Path(save_dir); outdir.mkdir(parents=True, exist_ok=True)
            for ext in formats:
                targets.append(outdir / f"{basename}.{ext.lstrip('.')}")
        else:
            targets = [save] if isinstance(save, (str, Path)) else list(save)
        for p in targets:
            p = Path(p)
            p.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(p, dpi=dpi, bbox_inches="tight")

    return (fig, ax, df) if return_handles else None
plot_intra_inter_hops_over_time(
    df,
    title="Station A ↔ Station B: Intra vs Inter hops (× grid)",
)
def plot_route_reliability_over_time(
    df,
    *,
    reliability_col,
    time_col="time",
    figsize=(10, 4),
    title=None,
    show=True,
    save=None,
    save_dir="figs",
    basename="route_reliability",
    formats=("png", "pdf"),
    dpi=300,
    return_handles=True,
):
    """
    只负责绘图。
    要求 df 中已经有 reliability_col 这一列。
    """
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from pathlib import Path

    reliability = df[reliability_col].astype(float)

    base_rc = {
        "font.family": "Times New Roman",
        "font.size": 14,
        "axes.labelsize": 18,
        "axes.titlesize": 18,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "axes.linewidth": 1.2,
    }

    plt.ion()
    with mpl.rc_context(base_rc):
        fig, ax = plt.subplots(figsize=figsize)

        ax.plot(
            df[time_col],
            reliability * 100,
            marker="o",
            markersize=3,
            linewidth=1.5,
            color="#4CAF50",
        )

        ax.set_xlabel("Time Step")
        ax.set_ylabel("Route Reliability (%)")
        ax.set_ylim(
            bottom=max(0, (reliability.min() * 100) - 2),
            top=min(100, (reliability.max() * 100) + 1),
        )

        if title is None:
            title = reliability_col
        ax.set_title(title)
        ax.grid(alpha=0.3, linestyle="--")
        plt.tight_layout()

        if show:
            plt.show(block=False)
            try:
                plt.pause(0.01)
            except Exception:
                pass

    if save:
        targets = []
        if save is True:
            outdir = Path(save_dir)
            outdir.mkdir(parents=True, exist_ok=True)
            for ext in formats:
                targets.append(outdir / f"{basename}.{ext.lstrip('.')}")
        else:
            targets = [save] if isinstance(save, (str, Path)) else list(save)

        for p in targets:
            p = Path(p)
            p.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(p, dpi=dpi, bbox_inches="tight")

    return (fig, ax) if return_handles else None


import  src.paper3_postprocess.route_reliable as route_reliable
rel_0999_099 = route_reliable.compute_route_reliability_series(
    df,
    p_intra=0.999,
    p_inter=0.99,
)

df[rel_0999_099.name] = rel_0999_099

fig, ax = plot_route_reliability_over_time(
    df,
    reliability_col=rel_0999_099.name,
    title="End-to-End Route Reliability",
    show=True,
)
df = df.copy()
df["rel_0999_099"] = rel_0999_099   # rel_0999_099 是你已算好的 Series
import sys
from src.config.viewer_config import ViewerConfig
import src.io.operate_group_data as operate_group_data

# station_a = int(df_s6_s8["station_a"].iloc[0])
# station_b = int(df_s6_s8["station_b"].iloc[0])
#
S6 = series_by_station[station_a]
S8 = series_by_station[station_b]

group_data_s6_s8 = operate_group_data.build_stationpair_group_data(
    S6, S8, WIN_START, WIN_END
)

PAIR_CFG = ViewerConfig(
    name=f"S{station_a}_S{station_b}_VERIFY",
    N=cfg.N,
    P=cfg.P,
    station_groups={
        0: {"name": f"S{station_a}", "stations": [station_a]},
        1: {"name": f"S{station_b}", "stations": [station_b]},
    },
    group_colors=["#ff4d4f", "#2f54eb"],
)

path_by_step = {}
for row in df_s6_s8.itertuples(index=False):
    p = getattr(row, "path", "")
    if isinstance(p, str) and p.strip():
        path_by_step[int(row.time)] = [int(x) for x in p.split("->")]

import time

t0 = time.perf_counter()
group_data_s6_s8 = operate_group_data.build_stationpair_group_data(
    S6, S8, WIN_START, WIN_END
)


print("build_stationpair_group_data:", time.perf_counter() - t0)

t0 = time.perf_counter()
path_by_step = {}
for row in df_s6_s8.itertuples(index=False):
    p = getattr(row, "path", "")
    if isinstance(p, str) and p.strip():
        path_by_step[int(row.time)] = [int(x) for x in p.split("->")]
print("build path_by_step:", time.perf_counter() - t0)

t0 = time.perf_counter()

viewer = SatelliteViewer(group_data_s6_s8, PAIR_CFG)
viewer.setWindowTitle(f"verify station{station_a}-station{station_b}")
viewer.resize(1400, 800)
viewer.static_edges = STATIC_EDGES
viewer.static_topology = True
viewer.set_paths(path_by_step)
viewer.show()
print("viewer part:", time.perf_counter() - t0)


build_stationpair_group_data: 1.8409442002885044
build path_by_step: 0.6856571002863348
viewer part: 0.44985050009563565
[cache] reusing cache from D:\paper3\data\satellitesposition\satellite_pos\_cache\cache_86164s_1s


{'step': 0,
 'time_s': 0.0,
 'playing': False,
 'timer_interval_ms': 200,
 'selected_sat_idx': None,
 'selected_sat_count': 0,
 'range_start_s': 0.0,
 'range_end_s': 86164.0,
 'num_steps': 86165,
 'num_sats': 648}

In [45]:
# ========== 路径 → intra/inter 链路分类 ==========
# 星座参数
# P = 18
# N = 36
# 1) 从 df 里拿当前 station pair


df_s6_s8 = df
station_a = int(df_s6_s8["station_a"].iloc[0])
station_b = int(df_s6_s8["station_b"].iloc[0])
dfpath = df["path"]


import src.model.get_intra_inter_link as get_intra_inter_link


all_intra = []
all_inter = []

for idx, path_str in enumerate(df["path"]):
    intra, inter = get_intra_inter_link.parse_path_links(path_str, N=N)
    all_intra.append(intra)
    all_inter.append(inter)

# 写回 DataFrame
df["intra_links"] = all_intra    # 每行是 [(src,dst), ...] 的 list
df["inter_links"] = all_inter

# 同时统计跳数
df["intra_hops"] = df["intra_links"].apply(len)
df["inter_hops"] = df["inter_links"].apply(len)
df["total_hops"] = df["intra_hops"] + df["inter_hops"]



import  src.paper3_postprocess.route_reliable as route_reliable
rel_0999_099 = route_reliable.compute_route_reliability_series(
    df,
    p_intra=0.999,
    p_inter=0.99,
)

df[rel_0999_099.name] = rel_0999_099


df = df.copy()
df["rel_0999_099"] = rel_0999_099   # rel_0999_099 是你已算好的 Series
import sys
from src.config.viewer_config import ViewerConfig
import src.io.operate_group_data as operate_group_data

# station_a = int(df_s6_s8["station_a"].iloc[0])
# station_b = int(df_s6_s8["station_b"].iloc[0])
#
S6 = series_by_station[station_a]
S8 = series_by_station[station_b]

group_data_s6_s8 = operate_group_data.build_stationpair_group_data(
    S6, S8, WIN_START, WIN_END
)

PAIR_CFG = ViewerConfig(
    name=f"S{station_a}_S{station_b}_VERIFY",
    N=cfg.N,
    P=cfg.P,
    station_groups={
        0: {"name": f"S{station_a}", "stations": [station_a]},
        1: {"name": f"S{station_b}", "stations": [station_b]},
    },
    group_colors=["#ff4d4f", "#2f54eb"],
)

path_by_step = {}
for row in df_s6_s8.itertuples(index=False):
    p = getattr(row, "path", "")
    if isinstance(p, str) and p.strip():
        path_by_step[int(row.time)] = [int(x) for x in p.split("->")]

import time

t0 = time.perf_counter()
group_data_s6_s8 = operate_group_data.build_stationpair_group_data(
    S6, S8, WIN_START, WIN_END
)


print("build_stationpair_group_data:", time.perf_counter() - t0)

t0 = time.perf_counter()
path_by_step = {}
for row in df_s6_s8.itertuples(index=False):
    p = getattr(row, "path", "")
    if isinstance(p, str) and p.strip():
        path_by_step[int(row.time)] = [int(x) for x in p.split("->")]
print("build path_by_step:", time.perf_counter() - t0)

t0 = time.perf_counter()

viewer = SatelliteViewer(group_data_s6_s8, PAIR_CFG)
viewer.setWindowTitle(f"verify station{station_a}-station{station_b}")
viewer.resize(1400, 800)
viewer.static_edges = STATIC_EDGES
viewer.static_topology = True
viewer.set_paths(path_by_step)
viewer.show()
_viewer_list.append(viewer)

print("viewer part:", time.perf_counter() - t0)


import src.paper3_postprocess.plot_hops_and_reliability_summary as plot_hops_and_reliability_summary

fig, axes, df_plot = plot_hops_and_reliability_summary.plot_hops_and_reliability_summary(
    df,
    reliability_col=rel_0999_099.name,
    title=f"Station {station_a} ↔ Station {station_b}: Hops and Reliability",
    hops_title="Intra / Inter / Total Hops",
    reliability_title="End-to-End Route Reliability",
    show=True,
    save=False,
    save_dir="figs/one_link_summary",
    basename=f"s{station_a}_s{station_b}_hops_reliability",
)

import importlib
import src.viz.vis3d as vis3d

importlib.reload(vis3d)

app, w = vis3d.show_globe_demo(
    P=18,
    N=36,
    start_s=WIN_START,
    end_s=WIN_END,
    auto_play=False,
    timer_interval_ms=200,
    initial_step=0,
    ephem_dir=Path(r"D:\paper3\data\basic_file\satellitesposition\satellite_pos"),
    cache_root=Path(r"D:\paper3\data\basic_file\satellitesposition\satellite_pos\_cache"),
    ignore_cache_source_dir=True,
)

w.set_paths(path_by_step)
w.jump_to_time(int(df_s6_s8["time"].iloc[0]))



build_stationpair_group_data: 0.2771427999250591
build path_by_step: 0.506175700109452
viewer part: 0.25225779972970486
[cache] reusing cache from D:\paper3\data\basic_file\satellitesposition\satellite_pos\_cache\cache_86164s_1s


{'step': 0,
 'time_s': 0.0,
 'playing': False,
 'timer_interval_ms': 200,
 'selected_sat_idx': None,
 'selected_sat_count': 0,
 'range_start_s': 0.0,
 'range_end_s': 86164.0,
 'num_steps': 86165,
 'num_sats': 648}

In [ ]:
# ephem_dir = r"D:\paper3\data\basic_file\satellitesposition\satellite_pos"
# cache_root = r"D:\paper3\data\satellitesposition\satellite_pos\_cache"


In [20]:
import src.paper3_postprocess.plot_hops_and_reliability_summary as plot_hops_and_reliability_summary

In [21]:
import importlib
importlib.reload(plot_hops_and_reliability_summary)

<module 'src.paper3_postprocess.plot_hops_and_reliability_summary' from 'D:\\paper3\\generic\\src\\paper3_postprocess\\plot_hops_and_reliability_summary.py'>

In [22]:
fig, axes, df_plot = plot_hops_and_reliability_summary.plot_hops_and_reliability_summary(
    df,
    reliability_col=rel_0999_099.name,
    title=f"Station {station_a} ↔ Station {station_b}: Hops and Reliability",
    hops_title="Intra / Inter / Total Hops",
    reliability_title="End-to-End Route Reliability",
    show=True,
    save=True,
    save_dir="figs/one_link_summary",
    basename=f"s{station_a}_s{station_b}_hops_reliability",
)


In [34]:
# import importlib
# import src.viz.vis3d as vis3d
#
# importlib.reload(vis3d)
#
# app, w = vis3d.show_globe_demo(
#     P=18,
#     N=36,
#     start_s=WIN_START,
#     end_s=WIN_END,
#     auto_play=False,
#     timer_interval_ms=200,
#     initial_step=0,
# )
#
# w.set_paths(path_by_step)
# w.jump_to_time(int(df_s6_s8["time"].iloc[0]))


[cache] reusing cache from D:\paper3\data\satellitesposition\satellite_pos\_cache\cache_86164s_1s


{'step': 0,
 'time_s': 0.0,
 'playing': False,
 'timer_interval_ms': 200,
 'selected_sat_idx': None,
 'selected_sat_count': 0,
 'range_start_s': 0.0,
 'range_end_s': 86164.0,
 'num_steps': 86165,
 'num_sats': 648}

In [ ]:
df.loc[df['time'] == 29651, ['time', 'station_a', 'station_b', 'path']]

In [22]:
import numpy as np

max_idx = np.argmin(rel_0999_099)
max_val = rel_0999_099[max_idx]

print("最大值下标:", max_idx)
print("最大值:", max_val)

最大值下标: 66605
最大值: 0.8908705074210587


In [ ]:
# import numpy as np
# import pandas as pd
#
# def reliability_global_stats(df, rel_col="rel_0999_099"):
#     x = pd.to_numeric(df[rel_col], errors="coerce").dropna()
#     return pd.Series({
#         "count": int(x.size),
#         "mean": float(x.mean()),
#         "median": float(x.median()),
#         "std": float(x.std(ddof=1)),
#         "min": float(x.min()),
#         "p05": float(x.quantile(0.05)),
#         "p10": float(x.quantile(0.10)),
#         "p90": float(x.quantile(0.90)),
#         "p95": float(x.quantile(0.95)),
#         "max": float(x.max()),
#         "time_ratio_rel_lt_0.95": float((x < 0.95).mean()),
#         "time_ratio_rel_lt_0.98": float((x < 0.98).mean()),
#         "time_ratio_rel_ge_0.99": float((x >= 0.99).mean()),
#     }, name=rel_col)
#
# def reliability_window_stats(df, rel_col="rel_0999_099", time_col="time", window_sec=300):
#     d = df[[time_col, rel_col]].copy()
#     d[time_col] = d[time_col].astype(int)
#     d[rel_col] = pd.to_numeric(d[rel_col], errors="coerce")
#     t0 = int(d[time_col].min())
#     d["win_id"] = ((d[time_col] - t0) // int(window_sec)).astype(int)
#     out = d.groupby("win_id", as_index=False).agg(
#         start_time=(time_col, "min"),
#         end_time=(time_col, "max"),
#         rel_mean=(rel_col, "mean"),
#         rel_min=(rel_col, "min"),
#         rel_p05=(rel_col, lambda s: s.quantile(0.05)),
#         rel_p95=(rel_col, lambda s: s.quantile(0.95)),
#     )
#     out["duration_sec"] = out["end_time"] - out["start_time"] + 1
#     return out
#
# def path_switch_stats(df, path_col="path", time_col="time"):
#     d = df[[time_col, path_col]].sort_values(time_col).copy()
#     d[path_col] = d[path_col].fillna("")
#     d["switch"] = d[path_col].ne(d[path_col].shift(1))
#     if len(d) > 0:
#         d.iloc[0, d.columns.get_loc("switch")] = False
#     d["seg_id"] = d["switch"].cumsum()
#     seg = d.groupby("seg_id", as_index=False).agg(
#         path=(path_col, "first"),
#         start_time=(time_col, "min"),
#         end_time=(time_col, "max"),
#     )
#     seg["duration_sec"] = seg["end_time"] - seg["start_time"] + 1
#     summary = pd.Series({
#         "path_switch_count": int(d["switch"].sum()),
#         "segment_count": int(len(seg)),
#         "avg_dwell_sec": float(seg["duration_sec"].mean()) if len(seg) else 0.0,
#         "median_dwell_sec": float(seg["duration_sec"].median()) if len(seg) else 0.0,
#         "max_dwell_sec": int(seg["duration_sec"].max()) if len(seg) else 0,
#     }, name="path_switch_summary")
#     return summary, seg


In [19]:
import pandas as pd
from pathlib import Path
import src.paper3_postprocess.route_statistic as route_statistic
global_stat = route_statistic.reliability_global_stats(df, rel_col="rel_0999_099")
win_5min = route_statistic.reliability_window_stats(df, rel_col="rel_0999_099", window_sec=300)
switch_summary, switch_segments = route_statistic.path_switch_stats(df, path_col="path")

print(global_stat)
print(win_5min.head())
print(switch_summary)


count                     86165.000000
mean                          0.926889
median                        0.935845
std                           0.018909
min                           0.890871
p05                           0.899869
p10                           0.900770
p90                           0.944353
p95                           0.945298
max                           0.951985
time_ratio_rel_lt_0.95        0.958278
time_ratio_rel_lt_0.98        1.000000
time_ratio_rel_ge_0.99        0.000000
Name: rel_0999_099, dtype: float64
   win_id  start_time  end_time  rel_mean   rel_min   rel_p05   rel_p95  \
0       0           0       299  0.941796  0.941523  0.941523  0.942465   
1       1         300       599  0.940949  0.940581  0.940581  0.941523   
2       2         600       899  0.941730  0.941523  0.941523  0.942465   
3       3         900      1199  0.946608  0.940581  0.941523  0.951033   
4       4        1200      1499  0.946866  0.941523  0.941523  0.951033   

   dura

In [124]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import src.model.get_intra_inter_link as get_intra_inter_link
import src.paper3_postprocess.read_path_csv as read_path_csv
import src.paper3_postprocess.route_reliable as route_reliable
import src.paper3_postprocess.route_statistic as route_statistic


def enrich_pair_df_with_reliability(
    df,
    *,
    N,
    p_intra=0.999,
    p_inter=0.99,
    rel_col="rel_0999_099",
    path_col="path",
):
    """
    对单个 pair 的时序 df 补齐:
    - intra_links / inter_links
    - intra_hops / inter_hops / total_hops
    - rel_col
    """
    df = df.copy()

    all_intra = []
    all_inter = []

    for path_str in df[path_col]:
        if isinstance(path_str, str) and path_str.strip():
            intra, inter = get_intra_inter_link.parse_path_links(path_str, N=N)
        else:
            intra, inter = [], []

        all_intra.append(intra)
        all_inter.append(inter)

    df["intra_links"] = all_intra
    df["inter_links"] = all_inter
    df["intra_hops"] = df["intra_links"].apply(len)
    df["inter_hops"] = df["inter_links"].apply(len)
    df["total_hops"] = df["intra_hops"] + df["inter_hops"]

    df[rel_col] = route_reliable.compute_route_reliability_series(
        df,
        p_intra=p_intra,
        p_inter=p_inter,
        intra_col="intra_hops",
        inter_col="inter_hops",
        name=rel_col,
    )

    # 空路径默认视为缺失，不参与 mean/p05 统计
    empty_mask = ~df[path_col].fillna("").astype(str).str.strip().astype(bool)
    df.loc[empty_mask, rel_col] = np.nan

    return df


def build_stationpair_reliability_stat_table(
    csv_dir,
    *,
    N,
    p_intra=0.999,
    p_inter=0.99,
    rel_col="rel_0999_099",
    save_dir=None,
    basename="stationpair_reliability_stat",
):
    """
    批量读取一个目录下的 pair CSV，
    复用 route_reliable + route_statistic，
    生成每个 station pair 的全局统计表。
    """
    csv_dir = Path(csv_dir)
    rows = []

    for csv_path in sorted(csv_dir.glob("*.csv")):
        df = read_path_csv.read_pair_csv(csv_path)
        if df.empty:
            continue

        station_a = int(df["station_a"].iloc[0])
        station_b = int(df["station_b"].iloc[0])

        df_enriched = enrich_pair_df_with_reliability(
            df,
            N=N,
            p_intra=p_intra,
            p_inter=p_inter,
            rel_col=rel_col,
        )

        global_stat = route_statistic.reliability_global_stats(
            df_enriched,
            rel_col=rel_col,
        )

        rec = {
            "pair_csv_name": csv_path.name,
            "station_a": station_a,
            "station_b": station_b,
        }
        rec.update(global_stat.to_dict())
        rows.append(rec)

    pair_stat_df = (
        pd.DataFrame(rows)
        .sort_values(["station_a", "station_b"])
        .reset_index(drop=True)
    )

    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        pair_stat_df.to_csv(
            save_dir / f"{basename}.csv",
            index=False,
            encoding="utf-8-sig",
        )

    return pair_stat_df


def build_stationpair_metric_matrix(
    pair_stat_df,
    *,
    metric="mean",
    station_a_order=None,
    station_b_order=None,
):
    """
    把 pair 统计表转成热力图矩阵。
    """
    if metric not in pair_stat_df.columns:
        raise ValueError(f"metric={metric!r} 不存在，可选列有: {pair_stat_df.columns.tolist()}")

    if station_a_order is None:
        station_a_order = sorted(pair_stat_df["station_a"].dropna().unique().tolist())
    if station_b_order is None:
        station_b_order = sorted(pair_stat_df["station_b"].dropna().unique().tolist())

    mat = pd.DataFrame(
        np.nan,
        index=station_a_order,
        columns=station_b_order,
        dtype=float,
    )

    for row in pair_stat_df.itertuples(index=False):
        mat.loc[int(row.station_a), int(row.station_b)] = float(getattr(row, metric))

    return mat


def plot_stationpair_metric_heatmap(
    pair_stat_df,
    *,
    metric="mean",
    station_a_order=None,
    station_b_order=None,
    figsize=(12, 8),
    title=None,
    cmap="viridis",
    vmin=None,
    vmax=None,
    annotate=False,
    fmt=".3f",
    x_group_breaks=None,
    y_group_breaks=None,
    show=True,
    save=False,
    save_dir="figs/heatmap",
    basename="stationpair_metric_heatmap",
    dpi=300,
    return_handles=True,
):
    """
    从 pair_stat_df 中取 metric 画热力图。
    同时把矩阵也保存成 csv，方便后续论文复用。
    """
    mat = build_stationpair_metric_matrix(
        pair_stat_df,
        metric=metric,
        station_a_order=station_a_order,
        station_b_order=station_b_order,
    )

    data = mat.to_numpy(dtype=float)

    if vmin is None:
        vmin = np.nanmin(data)
    if vmax is None:
        vmax = np.nanmax(data)

    cmap_obj = plt.get_cmap(cmap).copy()
    cmap_obj.set_bad(color="white")

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(
        data,
        aspect="auto",
        origin="upper",
        cmap=cmap_obj,
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_xticks(np.arange(len(mat.columns)))
    ax.set_xticklabels(mat.columns.tolist(), rotation=90)
    ax.set_yticks(np.arange(len(mat.index)))
    ax.set_yticklabels(mat.index.tolist())

    ax.set_xlabel("station_b")
    ax.set_ylabel("station_a")

    if title is None:
        title = f"Station-pair {metric} reliability matrix"
    ax.set_title(title)

    # 分块边界，可选；传的是“切分位置的索引”，不是 station id
    if x_group_breaks:
        for xb in x_group_breaks:
            ax.axvline(x=xb - 0.5, color="white", linewidth=1.5)
    if y_group_breaks:
        for yb in y_group_breaks:
            ax.axhline(y=yb - 0.5, color="white", linewidth=1.5)

    if annotate:
        for i in range(data.shape[0]):
            for j in range(data.shape[1]):
                val = data[i, j]
                if not np.isnan(val):
                    ax.text(
                        j, i, format(val, fmt),
                        ha="center", va="center",
                        fontsize=7, color="black",
                    )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(f"{metric} reliability")

    plt.tight_layout()

    if save:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)

        fig.savefig(save_dir / f"{basename}.png", dpi=dpi, bbox_inches="tight")
        mat.to_csv(save_dir / f"{basename}_matrix.csv", encoding="utf-8-sig")

    if show:
        plt.show(block=False)
        try:
            plt.pause(0.01)
        except Exception:
            pass

    return (fig, ax, mat) if return_handles else None


In [126]:
pair_stat_df = build_stationpair_reliability_stat_table(
    data_DIR,
    N=N,
    p_intra=0.999,
    p_inter=0.99,
    rel_col="rel_0999_099",
    save_dir="figs/heatmap",
    basename="gridx_stationpair_reliability_stat",
)

print(pair_stat_df.head())
print(pair_stat_df.columns.tolist())


                             pair_csv_name  station_a  station_b    count  \
0  region1--station0-region2--station5.csv          0          5  86165.0   
1  region1--station0-region2--station6.csv          0          6  86165.0   
2  region1--station0-region2--station7.csv          0          7  86165.0   
3  region1--station0-region2--station8.csv          0          8  86165.0   
4  region1--station0-region2--station9.csv          0          9  86165.0   

       mean    median       std       min       p05       p10       p90  \
0  0.951358  0.958676  0.017363  0.911732  0.920942  0.920942  0.969329   
1  0.947530  0.954847  0.014963  0.921864  0.922787  0.922787  0.960596   
2  0.960476  0.968359  0.021082  0.905328  0.913559  0.916305  0.978141   
3  0.946310  0.948140  0.012595  0.922787  0.931175  0.931175  0.959635   
4  0.947406  0.957717  0.014350  0.920942  0.921864  0.922787  0.959635   

        p95       max  time_ratio_rel_lt_0.95  time_ratio_rel_lt_0.98  \
0  0.969329  

In [127]:
fig, ax, mat = plot_stationpair_metric_heatmap(
    pair_stat_df,
    metric="mean",
    title="Station-pair mean reliability matrix",
    cmap="viridis",
    show=True,
    save=True,
    save_dir="figs/heatmap",
    basename="gridx_stationpair_mean_reliability",
)
